In [ ]:
from pathlib import Path
import polars as pl
from datetime import date, datetime, timezone
from zoneinfo import ZoneInfo

In [ ]:
DATA_FOLDER_PATH = Path("D:/TradingData/massive_flatfiles/us_stock_sip/minutes_agg_v1/2024/05")
DATE_TO_PICK = "2024-05-01"
DATA_PATH = DATA_FOLDER_PATH / f"{DATE_TO_PICK}.csv.gz"
mb = pl.read_csv(DATA_PATH)
mb

## Opening Box Scanner

This section builds a parameterized opening box for each ticker, joins prior daily statistics from the same flatfile folder, computes the same setup metrics used by the QuantConnect scanner, and sorts by `final_score`.

For Polygon/Massive minute aggregates, `window_start` is the bar start time in UTC nanoseconds. A `09:30` to `09:35` box therefore uses bars with starts `09:30`, `09:31`, `09:32`, `09:33`, and `09:34` in New York time.


In [ ]:


def parse_hhmm(value: str) -> int:
    hour, minute = map(int, value.split(":"))
    return hour * 60 + minute


def date_from_path(path: Path) -> date:
    return datetime.strptime(path.stem.removesuffix(".csv"), "%Y-%m-%d").date()


def market_time_to_utc_ns(scan_date: str, hhmm: str, timezone_name: str) -> int:
    hour, minute = map(int, hhmm.split(":"))
    market_tz = ZoneInfo(timezone_name)
    local_dt = datetime.strptime(scan_date, "%Y-%m-%d").replace(
        hour=hour,
        minute=minute,
        tzinfo=market_tz,
    )
    utc_dt = local_dt.astimezone(timezone.utc)
    return int(utc_dt.timestamp() * 1_000_000_000)


def add_bar_time_columns(frame, timezone_name: str):
    return frame.with_columns(
        pl.from_epoch("window_start", time_unit="ns")
        .dt.replace_time_zone("UTC")
        .dt.convert_time_zone(timezone_name)
        .alias("bar_time")
    )


def build_opening_box(
    minute_bars: pl.DataFrame,
    scan_date: str,
    box_start_time: str,
    box_end_time: str,
    timezone: str,
    min_box_bars: int,
) -> pl.DataFrame:
    start_ns = market_time_to_utc_ns(scan_date, box_start_time, timezone)
    end_ns = market_time_to_utc_ns(scan_date, box_end_time, timezone)

    return (
        minute_bars
        .filter(
            (pl.col("window_start") >= start_ns)
            & (pl.col("window_start") < end_ns)
        )
        .pipe(add_bar_time_columns, timezone)
        .sort(["ticker", "window_start"])
        .group_by("ticker")
        .agg(
            pl.first("open").alias("box_open"),
            pl.last("close").alias("box_close"),
            pl.max("high").alias("box_high"),
            pl.min("low").alias("box_low"),
            pl.sum("volume").alias("box_volume"),
            pl.sum("transactions").alias("box_transactions"),
            pl.count().alias("box_bar_count"),
            pl.min("bar_time").alias("box_first_bar_time"),
            pl.max("bar_time").alias("box_last_bar_time"),
        )
        .filter(pl.col("box_bar_count") >= min_box_bars)
    )

def prior_data_paths(data_folder: Path, scan_date: str, lookback_files: int) -> list[Path]:
    scan_day = datetime.strptime(scan_date, "%Y-%m-%d").date()
    dated_paths = []

    for candidate in data_folder.rglob("*.csv.gz"):
        try:
            candidate_day = date_from_path(candidate)
        except ValueError:
            continue

        if candidate_day < scan_day:
            dated_paths.append((candidate_day, candidate))

    return [path for _, path in sorted(dated_paths)[-lookback_files:]]


def load_prior_daily_stats(
    data_folder: Path,
    scan_date: str,
    timezone: str,
    lookback_files: int,
) -> pl.DataFrame:
    paths = prior_data_paths(data_folder, scan_date, lookback_files)

    if not paths:
        return pl.DataFrame(
            schema={
                "ticker": pl.String,
                "previous_close": pl.Float64,
                "avg_daily_volume_14": pl.Float64,
                "atr_14": pl.Float64,
                "daily_rows": pl.UInt32,
            }
        )

    scans = []

    for source_path in paths:
        session_date = date_from_path(source_path)
        scans.append(
            pl.scan_csv(source_path)
            .select("ticker", "volume", "open", "close", "high", "low", "window_start")
            .with_columns(pl.lit(session_date).alias("session_date"))
        )

    daily = (
        pl.concat(scans)
        .sort(["ticker", "session_date", "window_start"])
        .group_by(["ticker", "session_date"])
        .agg(
            pl.first("open").alias("daily_open"),
            pl.last("close").alias("daily_close"),
            pl.max("high").alias("daily_high"),
            pl.min("low").alias("daily_low"),
            pl.sum("volume").alias("daily_volume"),
        )
        .sort(["ticker", "session_date"])
        .with_columns(pl.col("daily_close").shift(1).over("ticker").alias("prior_daily_close"))
        .with_columns(
            pl.max_horizontal(
                pl.col("daily_high") - pl.col("daily_low"),
                (pl.col("daily_high") - pl.col("prior_daily_close")).abs(),
                (pl.col("daily_low") - pl.col("prior_daily_close")).abs(),
            ).alias("true_range")
        )
        .collect()
    )

    return (
        daily
        .group_by("ticker")
        .agg(
            pl.last("daily_close").alias("previous_close"),
            pl.col("daily_volume").tail(14).mean().alias("avg_daily_volume_14"),
            pl.col("true_range").drop_nulls().tail(14).mean().alias("atr_14"),
            pl.count().alias("daily_rows"),
        )
    )


def score_boxes(boxes: pl.DataFrame, daily_stats: pl.DataFrame) -> pl.DataFrame:
    scored = boxes.join(daily_stats, on="ticker", how="left")

    return (
        scored
        .with_columns(
            (pl.col("box_high") - pl.col("box_low")).alias("box_range"),
            ((pl.col("box_high") + pl.col("box_low")) / 2.0).alias("box_mid"),
            pl.when(pl.lit(EXPECTED_BOX_DAILY_VOLUME_SHARE) > 0)
            .then(pl.col("avg_daily_volume_14") * EXPECTED_BOX_DAILY_VOLUME_SHARE)
            .otherwise(None)
            .alias("expected_box_volume"),
        )
        .with_columns(
            pl.when(pl.col("expected_box_volume") > 0)
            .then(pl.col("box_volume") / pl.col("expected_box_volume"))
            .otherwise(None)
            .alias("orb_relative_volume"),
            pl.when(pl.col("atr_14") > 0)
            .then(pl.col("box_range") / pl.col("atr_14"))
            .otherwise(None)
            .alias("range_atr"),
            pl.when(pl.col("box_range") > 0)
            .then((pl.col("box_close") - pl.col("box_low")) / pl.col("box_range"))
            .otherwise(None)
            .alias("close_location"),
            pl.when(pl.col("box_range") > 0)
            .then((pl.col("box_close") - pl.col("box_open")).abs() / pl.col("box_range"))
            .otherwise(None)
            .alias("body_to_range"),
            pl.when(pl.col("previous_close") > 0)
            .then((pl.col("box_open") / pl.col("previous_close")) - 1.0)
            .otherwise(None)
            .alias("gap_pct"),
        )
        .with_columns(
            pl.when(pl.lit(IDEAL_RANGE_ATR_FRACTION) > 0)
            .then((1.0 - ((pl.col("range_atr") - IDEAL_RANGE_ATR_FRACTION).abs() / IDEAL_RANGE_ATR_FRACTION)).clip(0.0, 1.0))
            .otherwise(0.0)
            .alias("ideal_range_score"),
            pl.col("orb_relative_volume").fill_null(0.0).clip(0.0, 10.0).truediv(10.0).alias("rv_score"),
            pl.col("gap_pct").fill_null(0.0).clip(0.0, 0.10).truediv(0.10).alias("gap_score"),
            pl.when(pl.lit(LIQUIDITY_SCORE_VOLUME) > 0)
            .then((pl.col("avg_daily_volume_14") / LIQUIDITY_SCORE_VOLUME).clip(0.0, 1.0))
            .otherwise(0.0)
            .alias("liquidity_score"),
        )
        .with_columns(
            (
                35.0 * pl.col("rv_score")
                + 20.0 * pl.col("close_location").fill_null(0.0)
                + 15.0 * pl.col("gap_score")
                + 15.0 * pl.col("ideal_range_score")
                + 15.0 * pl.col("liquidity_score")
            ).alias("final_score")
        )
        .with_columns(
            (
                (pl.col("box_close") >= MIN_PRICE)
                & (pl.col("avg_daily_volume_14") >= MIN_AVG_DAILY_VOLUME)
                & (pl.col("atr_14") >= MIN_ATR)
                & (pl.col("orb_relative_volume") >= MIN_OPENING_RELATIVE_VOLUME)
                & (pl.col("box_high") > pl.col("box_low"))
                & (pl.col("gap_pct") >= MIN_GAP_UP_PCT)
                & (pl.col("box_close") > pl.col("box_open"))
                & (pl.col("range_atr") >= MIN_RANGE_ATR_FRACTION)
                & (pl.col("range_atr") <= MAX_RANGE_ATR_FRACTION)
                & (pl.col("close_location") >= MIN_CLOSE_LOCATION)
                & (pl.col("body_to_range") >= MIN_BODY_TO_RANGE)
                & (pl.col("final_score") >= MIN_SETUP_SCORE)
            ).fill_null(False).alias("passes_qc_setup_filter")
        )
        .with_columns(pl.col("final_score").fill_null(-1.0).alias("_sort_score"))
        .sort("_sort_score", descending=True)
        .drop("_sort_score")
    )




In [ ]:
# Read data 
SCAN_DATE = "2024-05-01"
SCAN_DATA_PATH = DATA_FOLDER_PATH / f"{SCAN_DATE}.csv.gz"
scan_minute_bars = pl.read_csv(SCAN_DATA_PATH)
scan_minute_bars

In [ ]:
# Scanner parameters. Change these to test other dates and opening-box windows.
BOX_START_TIME = "09:30"
BOX_END_TIME = "09:35"
MARKET_TIMEZONE = "America/New_York"
LOOKBACK_DAILY_FILES = 20
DAILY_LOOKBACK_ROOT = DATA_FOLDER_PATH.parent
MIN_BOX_BARS = 1

# QuantConnect scanner parameters mirrored here.
MIN_PRICE = 0.5
MIN_AVG_DAILY_VOLUME = 0
MIN_ATR = 0
EXPECTED_BOX_DAILY_VOLUME_SHARE = 0.0
MIN_OPENING_RELATIVE_VOLUME = 0
MIN_SETUP_SCORE = 0
MIN_GAP_UP_PCT = 0
MIN_CLOSE_LOCATION = 0
MIN_BODY_TO_RANGE = 0
MIN_RANGE_ATR_FRACTION = 0
MAX_RANGE_ATR_FRACTION = 100000
IDEAL_RANGE_ATR_FRACTION = 0
LIQUIDITY_SCORE_VOLUME = 0
# MIN_PRICE = 5.0
# MIN_AVG_DAILY_VOLUME = 100_000
# MIN_ATR = 0.50
# EXPECTED_BOX_DAILY_VOLUME_SHARE = 0.02
# MIN_OPENING_RELATIVE_VOLUME = 0.75
# MIN_SETUP_SCORE = 45.0
# MIN_GAP_UP_PCT = 0.005
# MIN_CLOSE_LOCATION = 0.60
# MIN_BODY_TO_RANGE = 0.20
# MIN_RANGE_ATR_FRACTION = 0.05
# MAX_RANGE_ATR_FRACTION = 0.80
# IDEAL_RANGE_ATR_FRACTION = 0.30
# LIQUIDITY_SCORE_VOLUME = 10_000_000


box_df = build_opening_box(
    scan_minute_bars,
    scan_date=SCAN_DATE,
    box_start_time=BOX_START_TIME,
    box_end_time=BOX_END_TIME,
    timezone=MARKET_TIMEZONE,
    min_box_bars=MIN_BOX_BARS,
)

daily_stats_df = load_prior_daily_stats(
    DAILY_LOOKBACK_ROOT,
    scan_date=SCAN_DATE,
    timezone=MARKET_TIMEZONE,
    lookback_files=LOOKBACK_DAILY_FILES,
)

scanner_df = score_boxes(box_df, daily_stats_df)

scanner_output_df = scanner_df.select(
    "ticker",
    "final_score",
    "passes_qc_setup_filter",
    "box_open",
    "box_high",
    "box_mid",
    "box_low",
    "box_close",
    "box_volume",
    "box_bar_count",
    "orb_relative_volume",
    "range_atr",
    "close_location",
    "body_to_range",
    "gap_pct",
    "previous_close",
    "avg_daily_volume_14",
    "atr_14",
    "daily_rows",
    "box_first_bar_time",
    "box_last_bar_time",
)

scanner_output_df
